# Модель для бота: какие вопросы задавать пользователю
Финальная модель (0.7860 на hold-out) использует около 350 признаков из четырёх таблиц, среди них внешние
рейтинги и агрегаты истории. Пользователь в чате такого не введёт. Поэтому для живого диалога нужна
упрощённая модель на признаках, которые человек может назвать сам.

Здесь сравниваем наборы вопросов по цене и пользе. Всё на dev, hold-out не трогаем.

Нужны: models/features_explainable.json (09), data/processed/oof_explainable.parquet (10).

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

sys.path.insert(0, "..")
from src.data import load_features, split_dev_holdout
from src.explain import EXCLUDED_GROUPS, build_mapping
from src.features import ID_COL, TARGET_COL
from src.metrics import calibration_report
from src.models import RANDOM_STATE, cv_predict, make_lgbm, to_lgbm

PROCESSED = Path("../data/processed")
REPORTS = Path("../reports")
REPORTS.mkdir(exist_ok=True)
pd.set_option("display.width", 160)

df = load_features(PROCESSED / "train_features.parquet")
dev, _ = split_dev_holdout(df, PROCESSED / "holdout_ids.csv")  # hold-out отбрасываем
y = dev[TARGET_COL]
features = json.loads((Path("../models") / "features_explainable.json").read_text())

## 1. Наборы признаков
- **Анкета**: то, что человек знает про себя и желаемый кредит. Отношения (кредит/доход, платёж/доход,
  срок, кредит/стоимость товара) бот считает сам.
- **Профессия**: тип профессии и организации (выбор кнопками). Это главные носители пола в данных, поэтому
  смотрим, сколько точности они дают.
- **История**: пять вопросов про кредиты в других банках. Они соответствуют агрегатам из `bureau`, но
  ответы человека лишь приближают настоящие значения.
- Внешние рейтинги `EXT_SOURCE_*` не спрашиваем: это анонимные оценки внешних источников, которых у человека нет.

In [ ]:
A_CORE = [
    "AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY", "AMT_GOODS_PRICE",
    "DAYS_EMPLOYED", "DAYS_EMPLOYED_ANOM", "NAME_INCOME_TYPE", "NAME_EDUCATION_TYPE",
    "FLAG_OWN_CAR", "OWN_CAR_AGE", "FLAG_OWN_REALTY", "NAME_CONTRACT_TYPE",
    "CREDIT_INCOME_RATIO", "ANNUITY_INCOME_RATIO", "CREDIT_TERM", "CREDIT_GOODS_RATIO",
]
PROFESSION = ["OCCUPATION_TYPE", "ORGANIZATION_TYPE"]
HISTORY = [
    "BUREAU_COUNT",                  # сколько всего кредитов брали в других банках
    "BUREAU_IS_ACTIVE_SUM",          # сколько из них сейчас активны
    "BUREAU_AMT_CREDIT_SUM_DEBT_SUM",  # сколько сейчас должны по ним в сумме
    "BUREAU_HAS_OVERDUE_SUM",        # были ли просрочки (число кредитов с просрочкой)
    "BUREAU_LAST1Y_COUNT",           # сколько кредитов взяли за последний год
    "HAS_BUREAU",                    # выводится из BUREAU_COUNT > 0
]
missing = [c for c in A_CORE + PROFESSION + HISTORY if c not in features]
assert not missing, f"нет среди объяснимых признаков: {missing}"

variants = {
    "A: анкета": (A_CORE, 11),
    "A + профессия": (A_CORE + PROFESSION, 13),
    "A + история": (A_CORE + HISTORY, 16),
    "A + профессия + история": (A_CORE + PROFESSION + HISTORY, 18),
}
for name, (cols, q) in variants.items():
    print(f"{name}: признаков {len(cols)}, вопросов пользователю около {q}")

## 2. Кросс-валидация (5 фолдов, те же, что везде)
Параметры скромные и не подбирались: признаков мало, нужна устойчивая простая модель.

In [ ]:
def make_bot_model():
    return make_lgbm(n_estimators=400, learning_rate=0.05, num_leaves=15, min_child_samples=100,
                     colsample_bytree=0.8, subsample=0.8, reg_lambda=3.0)


results = {}
for name, (cols, _) in variants.items():
    oof, fa = cv_predict(make_bot_model, to_lgbm(dev[cols]), y)
    results[name] = (oof, fa)
    print(f"{name}: AUC {fa.mean():.4f} ± {fa.std():.4f}")

In [ ]:
# Опорные точки: финальная модель (все объяснимые признаки), её OOF сохранён в 10
oof_ref = pd.read_parquet(PROCESSED / "oof_explainable.parquet")
assert (oof_ref[ID_COL].values == dev[ID_COL].values).all(), "порядок строк dev изменился"
skf = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)
fold_ref = np.array([roc_auc_score(y.iloc[va], oof_ref["baseline"].iloc[va]) for _, va in skf.split(dev, y)])
print(f"Финальная модель (все объяснимые признаки): AUC {fold_ref.mean():.4f} ± {fold_ref.std():.4f}")

## 3. Что получает пользователь на каждом наборе
При пороге 1/(1+R), R = 5: доля отказов, доля отсеянных дефолтов, экономия. Плюс калибровка и число
причин отказа, которые бот вообще может назвать.

In [ ]:
def decide(y_, p, R=5):
    t = 1 / (1 + R)
    reject = p >= t
    fn = int(((~reject) & (y_ == 1)).sum())
    fp = int((reject & (y_ == 0)).sum())
    loss = (R * fn + fp) / len(y_)
    naive = min(R * y_.mean(), 1 - y_.mean())
    return reject.mean(), ((reject) & (y_ == 1)).sum() / (y_ == 1).sum(), (1 - loss / naive) * 100


rows = []
yy = y.values
for name, (cols, q) in variants.items():
    oof, fa = results[name]
    refuse, recall, saving = decide(yy, oof)
    groups = set(build_mapping(cols).values) - EXCLUDED_GROUPS
    cal = calibration_report(name, yy, oof)
    rows.append({
        "набор": name, "вопросов": q, "AUC": round(fa.mean(), 4), "std": round(fa.std(), 4),
        "Gini": round(2 * fa.mean() - 1, 4), "средний прогноз": cal["mean pred"], "ECE": cal["ECE"],
        "отказов, %": round(refuse * 100, 1), "отсеяно дефолтов, %": round(recall * 100, 1),
        "экономия, %": round(saving, 1), "групп причин": len(groups),
    })
refuse, recall, saving = decide(yy, oof_ref["baseline"].values)
rows.append({
    "набор": "финальная (все объяснимые)", "вопросов": "около 350 признаков",
    "AUC": round(fold_ref.mean(), 4), "std": round(fold_ref.std(), 4), "Gini": round(2 * fold_ref.mean() - 1, 4),
    "средний прогноз": round(oof_ref["baseline"].mean(), 4), "ECE": np.nan,
    "отказов, %": round(refuse * 100, 1), "отсеяно дефолтов, %": round(recall * 100, 1),
    "экономия, %": round(saving, 1), "групп причин": len(set(build_mapping(features).values) - EXCLUDED_GROUPS),
})
table = pd.DataFrame(rows).set_index("набор")
table.to_csv(REPORTS / "bot_model_variants.csv")
table

## 4. Попарные разницы по фолдам
Сколько AUC даёт каждая добавка (профессия, история) относительно анкеты.

In [ ]:
base = results["A: анкета"][1]
for name in list(variants)[1:]:
    d = results[name][1] - base
    print(f"{name} - A: {d.mean():+.4f} (min {d.min():+.4f}, max {d.max():+.4f})")

d = results["A + профессия + история"][1] - results["A + история"][1]
print(f"\nПрофессия сверх истории: {d.mean():+.4f} (min {d.min():+.4f}, max {d.max():+.4f})")
d = results["A + профессия + история"][1] - results["A + профессия"][1]
print(f"История сверх профессии: {d.mean():+.4f} (min {d.min():+.4f}, max {d.max():+.4f})")

Что смотрим:
1. Насколько анкета сама по себе слабее финальной модели (разрыв измеряет ценность внешних рейтингов
   и агрегатов, которых у пользователя нет).
2. Сколько добавляют пять вопросов про историю и два про профессию, и стоят ли они усилий пользователя.
3. Остаётся ли модель откалиброванной (средний прогноз против 0.0807, ECE).
4. Что при этом происходит с отказами и экономией: слабая модель не должна обещать больше, чем может.